In [ ]:
import os,re,csv,warnings,unicodedata,hashlib,time, platform
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '12'   # OpenMP
os.environ['MKL_NUM_THREADS'] = '12' 
os.environ.setdefault('CHROMA_TELEMETRY_DISABLED', '1')
os.environ.setdefault('ANONYMIZED_TELEMETRY', 'false') 

from datetime import datetime
from typing import List, Tuple, Optional

import torch
torch.set_num_threads(12)         
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
transformers.logging.set_verbosity_info()

from docx import Document
from striprtf.striprtf import rtf_to_text

# ---------------- Optional: Polars for fast IO ----------------
try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False
    
try:
    import ahocorasick as aho
    HAS_AHO = True
except Exception:
    HAS_AHO = False

# ---------------- Optional: RAG stack (silently degrades if missing) -----------
RAG_DEPS_OK = True
try:
    import requests
    import trafilatura
    from langdetect import detect
    from sentence_transformers import SentenceTransformer
    from langchain_chroma import Chroma
except Exception:
    RAG_DEPS_OK = False

# ---------------- CUDA mem hygiene ----------------
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# ======================== CONFIG ========================
MODE = 'title'   # 'title' alebo 'keyword'

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'

CZ_LEXICON_TXT = "./Input/cz_legal_keywords.txt"   
CZ_LEXICON_CSV = "./Input/cz_legal_keywords.csv"   
CZ_HITS_THRESHOLD = 2
CZ_CHARS_BONUS    = 1     
MAX_REGEX_GROUP   = 256

THESAURUS_XLSX = "./Thesaurus/SK_Local_Thesaurus.xlsx"
THESAURUS_COL  = "slovak_terms"
STOPWORDS_TXT  = "./Input/stopword_dictionary.txt"

LLM_FALLBACK = "nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1"
LLM_PRIMARY  = "marcuscedricridia/8B-Nemotaur-IT"

MAX_NEW_TOKENS     = 128
BATCH_SIZE         = 4
DO_SAMPLE          = True
TEMPERATURE        = 0.1

TOP_P              = 0.9
MAX_TEXT_CHARS     = 8000
REPETITION_PENALTY = 2.0
MODE = "text-generation"

# --- RAG / Web corpus ---
RAG_ENABLE        = True            # zapne ak sú balíky dostupné
RAG_TOP_K         = 4               # počet pasáží pridávaných do promptu
RAG_MAX_DOCS_PER_DOMAIN = 300
RAG_MIN_CHARS     = 400
RAG_CHUNK_CHARS   = 900
RAG_CHUNK_OVERLAP = 120

SCRAPE_CACHE_DIR  = "./rag_cache"
INDEX_DIR         = "./rag_index"   # Chroma persist dir
ALLOWED_DOMAINS   = [
                    "www.slov-lex.sk",
                    "www.justice.gov.sk",
                    "www.zakonypreludi.sk",
                    ]
SEED_URLS = [
    "https://www.slov-lex.sk/pravne-predpisy",
    "https://www.justice.gov.sk/",
    "https://www.zakonypreludi.sk/"
]
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # CPU OK
SK_ENFORCE_REWRITE = True  

def configure_cuda_allocator():
    parts = ["max_split_size_mb:128"]
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)

configure_cuda_allocator()
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

# ======================== STOPWORDS ========================
def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, "r", encoding="utf-8") as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f"Loaded {len(words)} stopwords from {path}")
        return words
    except FileNotFoundError:
        print(f"No stopword file at {path}. Continuing without.")
        return set()
    except Exception as e:
        print(f"Failed to load stopwords: {e}")
        return set()

STOPWORDS = load_stopwords()

# ======================== Tezaurus I/O (Parquet/CSV prefer) ====================
def _ensure_fast_thesaurus(xlsx_path: str):
    if not HAS_POLARS:
        return
    pq_path = os.path.splitext(xlsx_path)[0] + ".parquet"
    csv_path = os.path.splitext(xlsx_path)[0] + ".csv"
    if os.path.exists(pq_path) and os.path.exists(csv_path):
        return
    try:
        df = pd.read_excel(xlsx_path, engine="openpyxl")
        pldf = pl.from_pandas(df)
        if not os.path.exists(pq_path):
            pldf.write_parquet(pq_path)
        if not os.path.exists(csv_path):
            pldf.write_csv(csv_path)
        print(f"[INFO] Thesaurus converted to {pq_path} and {csv_path}")
    except Exception as e:
        print(f"[WARN] Thesaurus conversion skipped: {e}")

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    _ensure_fast_thesaurus(xlsx_path)
    pq_path = os.path.splitext(xlsx_path)[0] + ".parquet"
    csv_path = os.path.splitext(xlsx_path)[0] + ".csv"

    if HAS_POLARS and os.path.exists(pq_path):
        df = pl.read_parquet(pq_path)
        if column not in df.columns:
                                raise ValueError(f'Column "{column}" not found in {pq_path}. Columns: {df.columns}')
        raw = df[column].drop_nulls().cast(pl.Utf8).to_list()
    elif HAS_POLARS and os.path.exists(csv_path):
        df = pl.read_csv(csv_path)
        if column not in df.columns:
                                raise ValueError(f'Column "{column}" not found in {csv_path}. Columns: {df.columns}')
        raw = df[column].drop_nulls().cast(pl.Utf8).to_list()
    else:
        if not os.path.exists(xlsx_path):
            raise FileNotFoundError(f"Thesaurus file not found: {xlsx_path}")
        df = pd.read_excel(xlsx_path, engine="openpyxl")
        if column not in df.columns:
            raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
        raw = df[column].dropna().astype(str).tolist()

    parts = []
    for cell in raw:
        parts.extend(re.split(r"[,\n;]+", str(cell)))

    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t:
            continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS:
            continue
        if t not in seen:
            seen.add(t)
            terms.append(t)

    print(f"Thesaurus terms loaded: {len(terms)}")
    return terms

# ======================== Slovakization fallback ===================
def load_cz_lexicon(txt_path=CZ_LEXICON_TXT, csv_path=CZ_LEXICON_CSV):
    exact, stems, regs = {}, {}, []

    def _add_exact(s, w): 
        if s: exact[s] = float(w) if w not in ("", None) else 1.0
    def _add_stem(s, w):
        s = s.rstrip("*").strip()
        if s: stems[s] = float(w) if w not in ("", None) else 1.0
    def _add_regex(p, w):
        try:
            regs.append((re.compile(p, re.IGNORECASE), float(w) if w not in ("", None) else 1.0))
        except re.error as e:
            print(f"[CZ] Bad regex skipped: /{p}/ ({e})")

    # CSV má prednosť
    if os.path.exists(csv_path):
        try:
            with open(csv_path, "r", encoding="utf-8") as f:
                rdr = csv.DictReader(f)
                for row in rdr:
                    term = (row.get("term") or "").strip()
                    typ  = (row.get("type") or "exact").strip().lower()
                    w    = row.get("weight")
                    if not term: 
                        continue
                    if typ == "regex":
                        _add_regex(term, w)
                    elif typ == "stem":
                        _add_stem(term, w)
                    else:
                        _add_exact(term, w)
            print(f"[CZ] CSV loaded: {len(exact)} exact, {len(stems)} stems, {len(regs)} regex.")
            return {"exact": exact, "stems": stems, "regex": regs}
        except Exception as e:
            print(f"[CZ] CSV load failed ({e}), trying TXT…")

    # TXT fallback
    if os.path.exists(txt_path):
        try:
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    s = line.strip()
                    if not s or s.startswith("#"):
                        continue
                    if s.lower().startswith("re:/") and s.endswith("/"):
                        _add_regex(s[3:-1], 1.0)
                    elif s.endswith("*"):
                        _add_stem(s[:-1].strip(), 1.0)
                    else:
                        _add_exact(s, 1.0)
            print(f"[CZ] TXT loaded: {len(exact)} exact, {len(stems)} stems, {len(regs)} regex.")
        except Exception as e:
            print(f"[CZ] TXT load failed: {e}")
    else:
        print("[CZ] No lexicon file found; CZ detection will rely on diacritics only.")

    return {"exact": exact, "stems": stems, "regex": regs}


# Regex fallback (iba ak Aho chýba), voliteľné – môžeš ponechať alebo ignorovať
def compile_cz_regex_fallback(lex):
    exact_res, stem_res = [], []
    if lex["exact"]:
        esc = [re.escape(w) for w in lex["exact"].keys()]
        for i in range(0, len(esc), MAX_REGEX_GROUP):
            group = esc[i:i+MAX_REGEX_GROUP]
            exact_res.append(re.compile(r"\b(?:" + "|".join(group) + r")\b", re.IGNORECASE))
    if lex["stems"]:
        esc = [re.escape(w) for w in lex["stems"].keys()]
        for i in range(0, len(esc), MAX_REGEX_GROUP):
            group = esc[i:i+MAX_REGEX_GROUP]
            stem_res.append(re.compile(r"\b(?:" + "|".join(group) + r")\w+", re.IGNORECASE))
    return exact_res, stem_res

def _chunked(iterable, n):
    it = list(iterable)
    for i in range(0, len(it), n):
        yield it[i:i+n]

def compile_cz_regexes(lex):
    """
    Vytvorí zoznam skompilovaných regexov:
        - exact_patterns: list[Pattern]
        - stem_patterns:  list[Pattern]
        - custom_patterns: list[Pattern]
    """
    exact_patterns, stem_patterns, custom_patterns = [], [], []

    # exact
    if lex["exact"]:
        ex_esc = [re.escape(w) for w in lex["exact"] if w]
        for group in _chunked(ex_esc, MAX_REGEX_GROUP):
            pat = r"\b(?:" + "|".join(group) + r")\b"
            exact_patterns.append(re.compile(pat, re.IGNORECASE))

    # stems (prefixy)
    if lex["stems"]:
        st_esc = [re.escape(w) for w in lex["stems"] if w]
        for group in _chunked(st_esc, MAX_REGEX_GROUP):
            pat = r"\b(?:" + "|".join(group) + r")\w+"
            stem_patterns.append(re.compile(pat, re.IGNORECASE))

    # vlastné regexy
    for raw in lex["regex"]:
        try:
            custom_patterns.append(re.compile(raw, re.IGNORECASE))
        except re.error as e:
            print(f"[CZ] Bad regex skipped: /{raw}/ ({e})")

    return exact_patterns, stem_patterns, custom_patterns

def build_aho_automata(lex):
    exact_automaton = None
    stem_automaton  = None

    if HAS_AHO and lex["exact"]:
        A = aho.Automaton()
        for term, w in lex["exact"].items():
            A.add_word(term, (term, float(w)))
        A.make_automaton()
        exact_automaton = A

    if HAS_AHO and lex["stems"]:
        B = aho.Automaton()
        for stem, w in lex["stems"].items():
            B.add_word(stem, (stem, float(w)))
        B.make_automaton()
        stem_automaton = B

    return exact_automaton, stem_automaton

def count_cz_hits(text: str) -> float:
    if not text:
        return 0.0
    score = 0.0

    # exact s presným boundary
    if A_EXACT is not None:
        for end_idx, (term, weight) in A_EXACT.iter(text):
            start_idx = end_idx - len(term) + 1
            if _is_word_boundary(text, start_idx, end_idx + 1):
                score += weight
    else:
        for rx in RX_EXACT_FB:
            for m in rx.finditer(text):
                w = float(CZ_LEXICON["exact"].get(m.group(0), 1.0))
                score += w

    # stem: prefix + aspoň 1 alnum za ním
    if A_STEM is not None:
        for end_idx, (stem, weight) in A_STEM.iter(text):
            start_idx = end_idx - len(stem) + 1
            e = end_idx + 1
            tail = e < len(text) and text[e].isalnum()
            left_ok = start_idx == 0 or not text[start_idx - 1].isalnum()
            if tail and left_ok:
                score += weight
    else:
        for rx in RX_STEM_FB:
            for m in rx.finditer(text):
                stem_word = m.group(0)
                w = 1.0
                for k, val in CZ_LEXICON["stems"].items():
                    if stem_word.lower().startswith(k.lower()):
                        w = float(val); break
                score += w

    for rx, w in RX_CUSTOM:
        score += len(rx.findall(text)) * float(w)

    return score

_CZ_CHARS_RE = re.compile(r"[ěĚřŘůŮ]")

CZ_LEXICON = load_cz_lexicon()
A_EXACT, A_STEM = build_aho_automata(CZ_LEXICON)
RX_EXACT_FB, RX_STEM_FB = compile_cz_regex_fallback(CZ_LEXICON)  # iba fallback
RX_CUSTOM = CZ_LEXICON["regex"]  # zoznam (compiled_re, weight)

def _is_word_boundary(s: str, start: int, end: int) -> bool:
    """Hard word-boundary: pred aj za matchom nie je alfanumerický znak (vrátane diakritiky)."""
    def _isalnum(ch):
        return ch.isalnum() or ch == "_"  # jednoduché; stačí na právne texty
    left_ok  = start == 0 or not _isalnum(s[start-1])
    right_ok = end   == len(s) or not _isalnum(s[end])
    return left_ok and right_ok

def is_probably_czech(s: str) -> bool:
    if not s:
        return False
    base = count_cz_hits(s)
    bonus = CZ_CHARS_BONUS if _CZ_CHARS_RE.search(s) else 0
    return (base + bonus) >= CZ_HITS_THRESHOLD
# ======================== Thesaurus matchery ========================
def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(" - ")[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r"^(.*?)(\s*\(([^)]+)\))\s*$", term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r"[\/,;]", inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", r"\\s+", re.escape(s))
    return s.replace(r"\-", r"[-–—]")

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r"\b" + _wildify(base) + r"(?:\s*\([^)]*\))?\b", re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r"\b" + _wildify(syn) + r"\b", re.IGNORECASE))
    pats_na = [re.compile(r"\b" + _wildify(strip_accents(base)) + r"(?:\s*\([^)]*\))?\b", re.IGNORECASE)]
    pats_na += [re.compile(r"\b" + _wildify(strip_accents(syn)) + r"\b", re.IGNORECASE) for syn in syns]
    return {"canon": canon, "patterns": pats, "patterns_noacc": pats_na, "synonyms": syns}

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))["canon"].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    hits = {}
    if not text:
        return []
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m["patterns"])
        b = sum(len(p.findall(t_na)) for p in m["patterns_noacc"])
        cnt = max(a, b)
        if cnt > 0:
            hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

# ======================== DOCX/RTF split ========================
_HEADING_NAME_PREFIXES = ("heading", "nadpis","po-heading-id")
_HEADING_EXACT_NAMES   = {"title","subtitle","toc heading","obsah","po-heading-id_"}
_PAGE_LINE_RE = re.compile(r"^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$", re.IGNORECASE)
_RUBRIC_RE    = re.compile(r"([.\-–—]{3,})")

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or "").strip().lower()
    except: name = ""
    try: sid  = (getattr(p.style,"style_id","") or "").lower()
    except: sid = ""
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r"^heading\d+$", name) or re.match(r"^nadpis\s*\d+$", name): return True
    if re.match(r"^heading\d+$", sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, "\n".join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, "\n".join(buf).strip()))
        return [("__DOCUMENT__", "\n".join(full).strip())] + sections
    except Exception as e:
        print(f"[WARN] DOCX split failed {path}: {e}")
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r"^\d+(\.\d+)*\s+\S", l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,"r",encoding="utf-8").read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, "\n".join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, "\n".join(buf).strip()))
        return [("__DOCUMENT__", "\n".join(full).strip())] + sections
    except Exception as e:
        print(f"[WARN] RTF split failed {path}: {e}")
        return []

# ======================== RAG: scraping + index + retrieve ======================
def _hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", "ignore")).hexdigest()

def _is_sk(text: str) -> bool:
    try:
        lang = detect(text[:4000])
        return lang == "sk"
    except Exception:
        return False

def _save_cache(path: str, text: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def _load_cache(path: str) -> Optional[str]:
    try:
        return open(path, "r", encoding="utf-8").read()
    except Exception:
        return None

def _chunk_text(s: str, size=RAG_CHUNK_CHARS, overlap=RAG_CHUNK_OVERLAP):
    s = re.sub(r"\n{3,}", "\n\n", (s or "").strip())
    if len(s) <= size:
        return [s] if len(s) >= RAG_MIN_CHARS else []
    chunks, i = [], 0
    while i < len(s):
        chunk = s[i:i+size]
        cut = chunk.rfind(". ")
        if cut > size//2:
            chunk = chunk[:cut+1]
        chunk = chunk.strip()
        if len(chunk) >= RAG_MIN_CHARS:
            chunks.append(chunk)
        i += max(1, size - overlap)
    return chunks

def crawl_domains(seed_urls=SEED_URLS, allowed_domains=ALLOWED_DOMAINS, limit_per_domain=RAG_MAX_DOCS_PER_DOMAIN):
    trafilatura.settings.use_browser = False
    seen, per_domain = set(), {d:0 for d in allowed_domains}
    from collections import deque
    q = deque(seed_urls)
    docs = []
    session = requests.Session()
    session.headers.update({"User-Agent": "LegalSK-RAG/1.0; polite crawler"})
    while q:
        url = q.popleft()
        from urllib.parse import urlparse, urljoin
        u = urlparse(url)
        if u.scheme not in ("http", "https"):
            continue
        netloc = u.netloc.lower()
        if netloc not in allowed_domains:
            continue
        if url in seen:
            continue
        seen.add(url)

        if per_domain.get(netloc, 0) >= limit_per_domain:
            continue

        cfile = os.path.join(SCRAPE_CACHE_DIR, netloc, _hash(url) + ".txt")
        cached = _load_cache(cfile)
        resp_text = None
        if cached:
            text = cached
        else:
            try:
                time.sleep(0.7)
                resp = session.get(url, timeout=10)
                if resp.status_code != 200 or "text/html" not in resp.headers.get("Content-Type",""):
                    continue
                resp_text = resp.text
                extracted = trafilatura.extract(resp_text, include_comments=False, include_tables=False)
                if not extracted:
                    continue
                text = extracted.strip()
                if not _is_sk(text) or len(text) < RAG_MIN_CHARS:
                    continue
                _save_cache(cfile, text)
            except Exception:
                continue

        per_domain[netloc] = per_domain.get(netloc, 0) + 1
        docs.append({"url": url, "domain": netloc, "text": text})

        # rozšír odkazy v rámci domény
        try:
            if resp_text is None:
                # ak sme brali z cache, už nevieme expandovať
                continue
            for m in re.finditer(r'href=["\"]([^"\"]+)["\"]', resp_text):
                nxt = urljoin(url, m.group(1))
                vu = urlparse(nxt)
                if vu.netloc.lower() == netloc and nxt not in seen:
                    q.append(nxt)
        except Exception:
            pass

    return docs

def build_or_load_index():
    if not (RAG_ENABLE and RAG_DEPS_OK):
        return None
    os.makedirs(INDEX_DIR, exist_ok=True)
    store = Chroma(collection_name="sk_legal_corpus", persist_directory=INDEX_DIR)
    existing = store.get().get("ids", [])
    if existing:
        return store

    # ak nič v indexe, zober cache alebo urob krátky crawl
    cache_found = False
    for d in ALLOWED_DOMAINS:
        ddir = os.path.join(SCRAPE_CACHE_DIR, d)
        if os.path.isdir(ddir) and any(fn.endswith(".txt") for fn in os.listdir(ddir)):
            cache_found = True
            break
    if not cache_found:
        print("[RAG] Crawling (first run)…")
        try:
            _ = crawl_domains()
        except Exception as e:
            print(f"[RAG] Crawl failed, continuing without RAG: {e}")
            return None

    # re-sken cache a chunkuj
    texts, metas, ids = [], [], []
    for d in ALLOWED_DOMAINS:
        ddir = os.path.join(SCRAPE_CACHE_DIR, d)
        if not os.path.isdir(ddir): 
            continue
        for fn in os.listdir(ddir):
            if not fn.endswith(".txt"):
                continue
            path = os.path.join(ddir, fn)
            txt = _load_cache(path)
            if not txt:
                continue
            for ch in _chunk_text(txt):
                texts.append(ch)
                metas.append({"domain": d, "path": path})
                ids.append(f"{fn}:{len(ids)}")

    if not texts:
        print("[RAG] No texts found for index.")
        return None

    # embeddingy (CPU)
    try:
        embed = SentenceTransformer(EMBED_MODEL, device="cpu")
        # Chroma sama zembedduje, ale môžeme jej nechať spraviť to interne
        store.add_texts(texts=texts, metadatas=metas, ids=ids)
        store.persist()
        print(f"[RAG] Indexed {len(texts)} chunks from cache.")
        return store
    except Exception as e:
        print(f"[RAG] Index build failed: {e}")
        return None

def retrieve_context(store, query: str, k: int = RAG_TOP_K) -> List[str]:
    if store is None:
        return []
    try:
        retriever = store.as_retriever(search_kwargs={"k": k})
        docs = retriever.invoke(query) or []
        ctx = []
        for d in docs:
            content = getattr(d, "page_content", "")
            if content:
                ctx.append(content.strip())
        return ctx[:k]
    except Exception:
        return []

# ======================== LLM loader ========================
def load_primary_llm():
    qconf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        LLM_PRIMARY,
        quantization_config=qconf,
        device_map=device_map,
        dtype=torch.float16,   # fp16 na GPU
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tok = AutoTokenizer.from_pretrained(LLM_PRIMARY)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    pipe = pipeline(
                    MODE,
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    repetition_penalty=REPETITION_PENALTY,
                    return_full_text=False,
                    )
    return pipe, f"primary:{LLM_PRIMARY}"

def load_fallback_llm():
    for k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_FLASH_ATTENTION"):
        if k in os.environ:
            os.environ.pop(k)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        LLM_FALLBACK,
        device_map=device_map,
        dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tok = AutoTokenizer.from_pretrained(LLM_FALLBACK)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    pipe = pipeline(
                    MODE,
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    repetition_penalty=REPETITION_PENALTY,
                    return_full_text=False,
                    )
    return pipe, f"fallback:{LLM_FALLBACK}"

try:
    LLM_PIPE, USED_MODEL = load_primary_llm()
    print(f"[LLM] Loaded {USED_MODEL}")
except Exception as e:
    print(f"[WARN] Primary model failed: {e}")
    LLM_PIPE, USED_MODEL = load_fallback_llm()
    print(f"[LLM] Loaded {USED_MODEL}")

# ======================== Prompting (s RAG kontextom) ========================
def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def prompt_title(text: str, th_priority: List[str], th_sample: List[str], ctx_snips: List[str]) -> str:
    pri  = "\n".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:40])
    ctx  = "\n\n".join(f"- {c.strip()}" for c in ctx_snips) if ctx_snips else "(bez kontextu)"
    examples = (
        "PRÍKLADY (formát):\n"
        "- "Náhrada škody pri porušení zmluvy"\n"
        "- "Správne konanie o uložení pokuty"\n"
        "- "Práva a povinnosti zamestnanca"\n\n"
    )
    return (
        "POUŽI VÝLUČNE SLOVENČINU (žiadna čeština ani čechizmy). "
        "ÚLOHA: Navrhni JEDEN presný a zrozumiteľný PRÁVNY NADPIS (3–12 slov) k textu nižšie.\n"
        "Požiadavky: spisovná slovenčina, vecný, bez bodky, bez úvodzoviek.\n"
        + examples +
        f"KONTEXT (RAG – slovenské zdroje):\n{ctx}\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "VÝSTUP (iba výsledný nadpis):"
    )

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str], ctx_snips: List[str]) -> str:
    pri  = ", ".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:40])
    ctx  = "\n".join(f"- {c.strip()}" for c in ctx_snips) if ctx_snips else "(bez kontextu)"
    return (
        "POUŽI VÝLUČNE SLOVENČINU (žiadna čeština ani čechizmy). "
        "ÚLOHA: Vyber JEDEN najvhodnejší PRÁVNY POJEM (1–4 slová) k nasledujúcemu textu.\n"
        "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky.\n\n"
        f"KONTEXT (RAG – slovenské zdroje):\n{ctx}\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "VÝSTUP (iba pojem):"
    )

# ======================== LLM volania (batched) ========================
def _clean_one_line(raw: str) -> str:
    txt = re.sub(r'^[\-\•\:\s]+', '', raw).strip()
    txt = re.sub(r'[\.，。…]+$', '', txt).strip()
    txt = txt.strip(' "„”')
    return txt.splitlines()[0].strip()

def _refine_to_slovak(text: str) -> str:
    refine_prompt = (
        "Prepíš nasledujúci text VÝLUČNE do spisovnej slovenčiny (žiadna čeština ani čechizmy). "
        "Výstup nech je bez bodky na konci a bez úvodzoviek. Vráť iba výsledný text.\n\n"
        f"TEXT:\n{text}\n\n"
        "VÝSTUP:"
    )
    out2 = LLM_PIPE(refine_prompt, num_return_sequences=1, do_sample=True, temperature=0.1, top_p=1.0, max_new_tokens=64)
    rec = out2[0] if isinstance(out2, list) else out2
    s = rec.get("generated_text", str(rec)) if isinstance(rec, dict) else str(rec)
    s = _clean_one_line(s)
    return s

def batched_generate_texts(prompts: List[str], batch_size: int = BATCH_SIZE) -> List[str]:
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out = LLM_PIPE(batch, batch_size=batch_size, num_return_sequences=1)
        for item in out:
            rec = item[0] if isinstance(item, list) else item
            raw = (
                rec.get("generated_text") if isinstance(rec, dict) else None
            ) or (
                rec.get("summary_text") if isinstance(rec, dict) else None
            ) or str(rec)
            cand = _clean_one_line(raw)
            if SK_ENFORCE_REWRITE and is_probably_czech(cand):
                cand = _refine_to_slovak(cand)
            outputs.append(cand)
    if len(outputs) != len(prompts):
        print(f"[WARN] batched_generate_texts: outputs={len(outputs)} != inputs={len(prompts)}")
    return outputs

# ======================== DRIVER ========================
def process_all(input_dir=INPUT_DIR):
    rows = []
    files = sorted(os.listdir(input_dir), key=str.lower)

    # RAG index (ak povolené a dostupné)
    store = build_or_load_index() if (RAG_ENABLE and RAG_DEPS_OK) else None
    if RAG_ENABLE and not RAG_DEPS_OK:
        print("[RAG] Dependencies missing -> RAG disabled.")
    if store is None:
        print("[RAG] No index available -> running LLM-only prompts.")

    # Priprav joby + bucketovanie podľa dĺžky promptu
    jobs = []   # (mode, file, header, prio_terms, prompt)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith(".docx"):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith(".rtf"):
            secs = split_rtf_by_headers(p)
        else:
            continue

        if not secs:
            print(f"No text in {f}")
            continue

        for header, text in secs:
            if not text.strip():
                continue

            short = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:40]

            ctx_snips = retrieve_context(store, short, k=RAG_TOP_K) if store else []

            if MODE == "title":
                prompt = prompt_title(short, prio_terms, sample_terms, ctx_snips)
                jobs.append(("title", f, header, prio_terms, prompt))
            else:
                prompt = prompt_keyword(short, prio_terms, sample_terms, ctx_snips)
                jobs.append(("keyword", f, header, prio_terms, prompt))

        print(f"Queued {f} with {len(secs)} sections.")

    if not jobs:
        print("No jobs queued.")
        return pd.DataFrame([])

    # Bucketovanie podľa dĺžky promptu
    jobs.sort(key=lambda j: len(j[-1]))

    prompts = [j[-1] for j in jobs]
    gens = batched_generate_texts(prompts, batch_size=BATCH_SIZE)

    for (mode, f, header, prio_terms, _), gen in zip(jobs, gens):
        if mode == "title":
            rows.append({
                "file": f,
                "header": header,
                "suggested_title": gen,
                "method": f"llm{"-rag" if store else ""} ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })
        else:
            rows.append({
                "file": f,
                "header": header,
                "top_keyword": gen,
                "method": f"llm{"-rag" if store else ""} ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })

    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if MODE == "title":
        df = pd.DataFrame(rows, columns=["file","header","suggested_title","method","priority_terms"])
        stub = "llm_rag_titles" if store else "llm_only_titles"
    else:
        df = pd.DataFrame(rows, columns=["file","header","top_keyword","method","priority_terms"])
        stub = "llm_rag_keywords" if store else "llm_only_keywords"

    csv_path  = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")

    return df

if __name__ == "__main__":
    df = process_all(INPUT_DIR)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))

SyntaxError: invalid character '„' (U+201E) (3998017210.py, line 826)